# Helene GWR — 100mi cutoff (Cluster-Level, N=101)

Companion to [`helene_gwr.ipynb`](helene_gwr.ipynb) which used the 50-mi cutoff (N=38). Once the 50-mi-cutoff result was identified as a sample-selection artefact (see [findings.md](../notes/findings.md) §"Affected-region cutoff sensitivity"), this notebook re-runs GWR on the wider 100-mi Helene footprint where:

- N = 101 clusters (was 38) — finally adequate sample for GWR/MGWR
- Buncombe NC (Asheville) is in cluster 17 — the Appalachian disaster region is properly represented
- dist_to_track range: 1 – 117 mi (was 3 – 61 mi)
- The kernel-edge artefact I worried about in the 50-mi notebook (bandwidth k=36/38, essentially global) should resolve with N=101

**Same design choices as `helene_gwr.ipynb`:**
- Cluster-level (not raw counties — SARIMAX baselines are noisier at county scale)
- Euclidean cluster-centroid distance kernel
- Bisquare kernel, AICc adaptive bandwidth
- 4 IVs: `median_household_income`, `pct_white`, `nchs_code`, `dist_to_track_mi`
- `insurance_coverage_pct` dropped (saturated, see findings.md sensitivity §)

**Question to answer:**
Does the dist_to_track_mi β still show coastal-vs-inland localization (as I claimed in the 50-mi notebook), or does it become spatially homogeneous now that the truncation is corrected? My current best guess is the latter — the spatial heterogeneity I described was likely a kernel-edge artefact from N=38.

In [1]:
import os, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

from libpysal.weights import Queen, KNN
from esda.moran import Moran
from mgwr.gwr import GWR, MGWR
from mgwr.sel_bw import Sel_BW

warnings.filterwarnings("ignore")

OUTPUT_DIR = "../results/helene_gwr_100mi/"
FIG_DIR    = os.path.join(OUTPUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)
print(f"Output dir: {OUTPUT_DIR}")

Output dir: ../results/helene_gwr_100mi/


/Users/qing/miniconda3/envs/geo/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## §1. Load and prep

In [2]:
# Helene 100mi cluster metrics from pooled_dataset_100mi.csv
pooled = pd.read_csv("../results/local_level/regression/pooled_dataset_100mi.csv")
helene = pooled[pooled["hurricane"] == "helene"].copy().reset_index(drop=True)
helene["cluster"] = helene["NAME"].str.replace("Cluster_", "").astype(int)
print(f"Helene 100mi clusters: N={len(helene)}")

# County→cluster + county shapes (100mi cutoff)
ca = pd.read_csv("../results/local_level/helene_100mi/county_cluster_assignments.csv")
ca["GEOID"] = ca["GEOID"].astype(int)
county_shp = "./../../hurricane_oct/data/county_geo/tl_2023_us_county/tl_2023_us_county.shp"
counties_gdf = gpd.read_file(county_shp)
counties_gdf["GEOID"] = counties_gdf["GEOID"].astype(int)
gdf_counties = counties_gdf[counties_gdf["GEOID"].isin(ca["GEOID"])].copy()
gdf_counties = gdf_counties.merge(ca[["GEOID", "cluster"]], on="GEOID", how="left")
gdf_counties = gdf_counties.to_crs(epsg=5070)
gdf_clusters = gdf_counties.dissolve(by="cluster")[["geometry"]].reset_index()
gdf = gdf_clusters.merge(helene, on="cluster", how="left")
centroids = gdf.geometry.centroid
gdf["centroid_x"] = centroids.x
gdf["centroid_y"] = centroids.y

from shapely.geometry import LineString
track_gdf = gpd.read_file("./../../hurricane_oct/data/storm_track/helene_storm_track.shp").to_crs(epsg=5070)
if track_gdf.geometry.geom_type.iloc[0] == "Point":
    track_line = LineString(track_gdf.geometry.tolist())
else:
    track_line = track_gdf.unary_union
track_line_gdf = gpd.GeoDataFrame(geometry=[track_line], crs="EPSG:5070")

# Verify Buncombe NC is in the GeoDataFrame
buncombe_cluster = ca[ca["GEOID"] == 37021]["cluster"].iloc[0]
print(f"Buncombe NC (Asheville) is in cluster {buncombe_cluster}")
print(f"Cluster {buncombe_cluster} dist_to_track: {gdf[gdf['cluster']==buncombe_cluster]['dist_to_track_mi'].iloc[0]:.1f} mi")

Helene 100mi clusters: N=101
Buncombe NC (Asheville) is in cluster 17
Cluster 17 dist_to_track: 62.3 mi


## §2. Global OLS baseline

In [3]:
FEATURES = ["median_household_income", "pct_white", "nchs_code", "dist_to_track_mi"]
DVs = [("largest_drop_within", "Largest Drop — Within (%)"),
       ("recovery_days_within", "Recovery Time — Within (days)")]

valid_mask = gdf[FEATURES + [d[0] for d in DVs]].notna().all(axis=1)
gdf_model = gdf[valid_mask].copy().reset_index(drop=True)
print(f"After dropping NaN: N={len(gdf_model)} (was {len(gdf)})")

scaler = StandardScaler()
X_z = scaler.fit_transform(gdf_model[FEATURES])
X_z_df = pd.DataFrame(X_z, columns=FEATURES, index=gdf_model.index)
for f in FEATURES:
    gdf_model[f + "_z"] = X_z_df[f]

ols_results = {}
for dv_col, dv_label in DVs:
    y = gdf_model[dv_col].values
    X = sm.add_constant(X_z_df.values)
    m = sm.OLS(y, X).fit()
    ols_results[dv_col] = m
    print(f"\n{'='*70}\nOLS: {dv_label}\n{'='*70}")
    print(f"R²={m.rsquared:.3f}  Adj.R²={m.rsquared_adj:.3f}  F p={m.f_pvalue:.4f}")
    print(f"  {'Variable':<28} {'β':>8} {'p':>8}")
    print("  " + "-"*46)
    for n, b, p in zip(["const"] + FEATURES, m.params, m.pvalues):
        sig = "**" if p < 0.05 else "*" if p < 0.10 else ""
        print(f"  {n:<28} {b:>8.3f} {p:>8.4f} {sig}")

After dropping NaN: N=101 (was 101)

OLS: Largest Drop — Within (%)
R²=0.048  Adj.R²=0.008  F p=0.3161
  Variable                            β        p
  ----------------------------------------------
  const                         -15.213   0.0000 **
  median_household_income         1.209   0.1033 
  pct_white                       0.105   0.8732 
  nchs_code                       0.056   0.9405 
  dist_to_track_mi               -0.643   0.2954 

OLS: Recovery Time — Within (days)
R²=0.004  Adj.R²=-0.037  F p=0.9819
  Variable                            β        p
  ----------------------------------------------
  const                           4.719   0.0000 **
  median_household_income         0.035   0.8883 
  pct_white                       0.030   0.8936 
  nchs_code                       0.033   0.8963 
  dist_to_track_mi                0.116   0.5759 


## §3. Spatial diagnostics gate

In [4]:
w_queen = Queen.from_dataframe(gdf_model)
w_knn = KNN.from_dataframe(gdf_model, k=4)
for w_, label in [(w_queen, "Queen"), (w_knn, "KNN(k=4)")]:
    print(f"{label}: n={w_.n}, mean_neighbors={w_.mean_neighbors:.1f}, "
          f"min/max={w_.min_neighbors}/{w_.max_neighbors}")

diag_rows = []
for dv_col, dv_label in DVs:
    resid = ols_results[dv_col].resid
    print(f"\n--- {dv_label} ---")
    for w_, wlabel in [(w_queen, "Queen"), (w_knn, "KNN4")]:
        try:
            mi = Moran(resid, w_, permutations=999)
            print(f"  Moran's I ({wlabel}): I={mi.I:+.3f}, p_sim={mi.p_sim:.3f}")
            diag_rows.append({"dv": dv_label, "test": f"Moran_{wlabel}",
                              "stat": round(mi.I, 4), "p": round(mi.p_sim, 4)})
        except Exception as e:
            print(f"  Moran's I ({wlabel}) failed: {e}")
    rho_x, p_x = spearmanr(gdf_model["centroid_x"], resid)
    rho_y, p_y = spearmanr(gdf_model["centroid_y"], resid)
    print(f"  Spearman resid vs centroid_x: ρ={rho_x:+.3f}, p={p_x:.3f}")
    print(f"  Spearman resid vs centroid_y: ρ={rho_y:+.3f}, p={p_y:.3f}")
    diag_rows.append({"dv": dv_label, "test": "Spearman_resid_x",
                      "stat": round(rho_x, 4), "p": round(p_x, 4)})
    diag_rows.append({"dv": dv_label, "test": "Spearman_resid_y",
                      "stat": round(rho_y, 4), "p": round(p_y, 4)})

diag_df = pd.DataFrame(diag_rows)
diag_df.to_csv(os.path.join(OUTPUT_DIR, "spatial_diagnostics.csv"), index=False)
PROCEED_GWR = bool((diag_df["p"].values < 0.10).any())
print(f"\nPROCEED_GWR = {PROCEED_GWR}")

Queen: n=101, mean_neighbors=4.5, min/max=1/18
KNN(k=4): n=101, mean_neighbors=4.0, min/max=4/4

--- Largest Drop — Within (%) ---
  Moran's I (Queen): I=+0.054, p_sim=0.173
  Moran's I (KNN4): I=+0.022, p_sim=0.294
  Spearman resid vs centroid_x: ρ=+0.108, p=0.282
  Spearman resid vs centroid_y: ρ=-0.104, p=0.300

--- Recovery Time — Within (days) ---
  Moran's I (Queen): I=-0.016, p_sim=0.491
  Moran's I (KNN4): I=-0.023, p_sim=0.468
  Spearman resid vs centroid_x: ρ=-0.062, p=0.540
  Spearman resid vs centroid_y: ρ=+0.048, p=0.637

PROCEED_GWR = False


## §4. GWR fit

With N=101 we don't need the band-aid `bw_min` from the 50-mi notebook; mgwr's default lower bound (40 + 2·n_vars = 50) is now strictly less than N. Still pass explicit bounds for transparency.

In [5]:
gwr_results = {}
if not PROCEED_GWR:
    print("PROCEED_GWR=False — skipping GWR fit.")
else:
    coords = list(zip(gdf_model["centroid_x"].values, gdf_model["centroid_y"].values))
    X_mat = X_z_df.values
    n_vars_eff = X_mat.shape[1] + 1
    bw_min_adapt = max(n_vars_eff + 3, 8)
    bw_max_adapt = len(gdf_model) - 1
    print(f"Adaptive bandwidth search bounds: [{bw_min_adapt}, {bw_max_adapt}]")
    for dv_col, dv_label in DVs:
        y = gdf_model[[dv_col]].values
        print(f"\n{'='*70}\nGWR: {dv_label}\n{'='*70}")
        sel = Sel_BW(coords, y, X_mat, fixed=False, kernel="bisquare")
        bw = sel.search(criterion="AICc", bw_min=bw_min_adapt, bw_max=bw_max_adapt)
        print(f"  Optimal adaptive bandwidth (k neighbors): {bw}")
        try:
            gwr_mod = GWR(coords, y, X_mat, bw, fixed=False, kernel="bisquare").fit()
            print(f"  AICc: {gwr_mod.aicc:.2f}  R²: {gwr_mod.R2:.3f}  Adj.R²: {gwr_mod.adj_R2:.3f}")
            print(f"  Effective n. parameters: {gwr_mod.ENP:.1f}")
            print(f"  Local R² range: [{gwr_mod.localR2.min():.3f}, {gwr_mod.localR2.max():.3f}]")
            gwr_results[dv_col] = {"model": gwr_mod, "bw": bw, "selector": sel}
        except Exception as e:
            print(f"  GWR fit failed: {e}")
            gwr_results[dv_col] = None

PROCEED_GWR=False — skipping GWR fit.


## §5. MGWR (likely to actually work now)

In [6]:
mgwr_results = {}
if not PROCEED_GWR:
    print("PROCEED_GWR=False — skipping MGWR fit.")
else:
    coords = list(zip(gdf_model["centroid_x"].values, gdf_model["centroid_y"].values))
    X_mat = X_z_df.values
    n_vars_eff = X_mat.shape[1] + 1
    mbw_min = max(n_vars_eff + 3, 8)
    mbw_max = len(gdf_model) - 1
    print(f"MGWR per-predictor bandwidth bounds: [{mbw_min}, {mbw_max}] for {n_vars_eff} coefs")
    for dv_col, dv_label in DVs:
        y = gdf_model[[dv_col]].values
        print(f"\n{'='*70}\nMGWR: {dv_label}\n{'='*70}")
        try:
            sel = Sel_BW(coords, y, X_mat, multi=True, fixed=False, kernel="bisquare")
            bws = sel.search(criterion="AICc",
                              multi_bw_min=[mbw_min] * n_vars_eff,
                              multi_bw_max=[mbw_max] * n_vars_eff)
            print(f"  Per-covariate adaptive bandwidths: {bws}")
            mgwr_mod = MGWR(coords, y, X_mat, sel, fixed=False, kernel="bisquare").fit()
            print(f"  AICc: {mgwr_mod.aicc:.2f}  R²: {mgwr_mod.R2:.3f}  Adj.R²: {mgwr_mod.adj_R2:.3f}")
            print(f"  Local R² range: [{mgwr_mod.localR2.min():.3f}, {mgwr_mod.localR2.max():.3f}]")
            mgwr_results[dv_col] = {"model": mgwr_mod, "bws": bws, "selector": sel}
        except Exception as e:
            print(f"  MGWR failed: {e}")
            mgwr_results[dv_col] = None

PROCEED_GWR=False — skipping MGWR fit.


## §6. Diagnostics — AICc comparison + Monte Carlo non-stationarity

In [7]:
def ols_aicc(m, n):
    k = m.df_model + 1
    return m.aic + (2 * k * (k + 1)) / max(n - k - 1, 1)

n = len(gdf_model)
comp_rows = []
for dv_col, dv_label in DVs:
    ols_m = ols_results[dv_col]
    row = {"dv": dv_label,
           "OLS_AICc": round(ols_aicc(ols_m, n), 2),
           "OLS_R2": round(ols_m.rsquared, 3),
           "OLS_adjR2": round(ols_m.rsquared_adj, 3)}
    if PROCEED_GWR and gwr_results.get(dv_col):
        gm = gwr_results[dv_col]["model"]
        row.update({"GWR_AICc": round(gm.aicc, 2), "GWR_R2": round(gm.R2, 3),
                    "GWR_adjR2": round(gm.adj_R2, 3), "GWR_bw": gwr_results[dv_col]["bw"]})
    if PROCEED_GWR and mgwr_results.get(dv_col):
        mm = mgwr_results[dv_col]["model"]
        row.update({"MGWR_AICc": round(mm.aicc, 2), "MGWR_R2": round(mm.R2, 3),
                    "MGWR_adjR2": round(mm.adj_R2, 3)})
    comp_rows.append(row)

comp_df = pd.DataFrame(comp_rows)
comp_df.to_csv(os.path.join(OUTPUT_DIR, "model_comparison_aicc.csv"), index=False)
print("Model comparison (lower AICc is better):")
print(comp_df.to_string(index=False))

# Monte Carlo non-stationarity test
if PROCEED_GWR:
    nsta_rows = []
    for dv_col, dv_label in DVs:
        if not gwr_results.get(dv_col):
            continue
        gm = gwr_results[dv_col]["model"]
        sel = gwr_results[dv_col]["selector"]
        print(f"\n--- Non-stationarity test: {dv_label} ---")
        try:
            pvals = gm.spatial_variability(sel, n_iters=200)
            for nm, pv in zip(["intercept"] + FEATURES, pvals):
                sig = "**" if pv < 0.05 else "*" if pv < 0.10 else ""
                print(f"  {nm:<28} p={float(pv):.3f} {sig}")
                nsta_rows.append({"dv": dv_label, "variable": nm,
                                  "p_nonstationarity": round(float(pv), 4)})
        except Exception as e:
            print(f"  spatial_variability failed: {e}")
    if nsta_rows:
        pd.DataFrame(nsta_rows).to_csv(
            os.path.join(OUTPUT_DIR, "nonstationarity_test.csv"), index=False)

Model comparison (lower AICc is better):
                           dv  OLS_AICc  OLS_R2  OLS_adjR2
    Largest Drop — Within (%)    655.04   0.048      0.008
Recovery Time — Within (days)    436.36   0.004     -0.037


## §7. Maps — Local R² + β + t-stat

In [ ]:
def plot_gwr_maps(gdf_in, gwr_mod, dv_label, dv_key, feat_names, fig_dir, track_gdf):
    gp = gdf_in.copy()
    gp["local_R2"] = gwr_mod.localR2.flatten()
    for i, fn in enumerate(["intercept"] + feat_names):
        gp[f"beta_{fn}"] = gwr_mod.params[:, i]
        gp[f"tval_{fn}"] = gwr_mod.tvalues[:, i]
    n_pred = len(feat_names)
    fig, ax = plt.subplots(figsize=(10, 8))
    gp.plot(column="local_R2", cmap="viridis", legend=True,
            edgecolor="black", linewidth=0.3, ax=ax,
            legend_kwds={"label": "Local R²", "shrink": 0.6})
    track_gdf.plot(ax=ax, color="red", linewidth=2)
    ax.set_title(f"GWR Local R² — {dv_label} (100mi N={len(gp)})", fontweight="bold"); ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, f"gwr_localR2_{dv_key}.png"), dpi=150, bbox_inches="tight")
    plt.show()
    fig, axes = plt.subplots(2, n_pred, figsize=(5*n_pred, 10))
    if n_pred == 1: axes = axes.reshape(-1, 1)
    for j, fn in enumerate(feat_names):
        ax = axes[0, j]
        v = np.abs(gp[f"beta_{fn}"]).max()
        gp.plot(column=f"beta_{fn}", cmap="RdBu_r", vmin=-v, vmax=v, legend=True,
                edgecolor="black", linewidth=0.3, ax=ax,
                legend_kwds={"label": f"β({fn})", "shrink": 0.6})
        track_gdf.plot(ax=ax, color="black", linewidth=1.5)
        ax.set_title(f"β: {fn}", fontweight="bold"); ax.set_axis_off()
        ax = axes[1, j]
        gp["_sig"] = np.where(np.abs(gp[f"tval_{fn}"]) > 1.96, gp[f"tval_{fn}"], np.nan)
        v2 = max(np.nanmax(np.abs(gp["_sig"])) if gp["_sig"].notna().any() else 0, 2)
        gp.plot(column="_sig", cmap="RdBu_r", vmin=-v2, vmax=v2,
                missing_kwds={"color": "lightgrey"}, legend=True,
                edgecolor="black", linewidth=0.3, ax=ax,
                legend_kwds={"label": f"t({fn}) |t|>1.96", "shrink": 0.6})
        track_gdf.plot(ax=ax, color="black", linewidth=1.5)
        ax.set_title(f"t-stat sig: {fn}", fontweight="bold"); ax.set_axis_off()
    fig.suptitle(f"GWR coefficient & t-stat maps — {dv_label} (100mi N={len(gp)})", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, f"gwr_coef_tstat_{dv_key}.png"), dpi=150, bbox_inches="tight")
    plt.show()
    return gp

if PROCEED_GWR:
    for dv_col, dv_label in DVs:
        if gwr_results.get(dv_col):
            plot_gwr_maps(gdf_model, gwr_results[dv_col]["model"], dv_label,
                          dv_col, FEATURES, FIG_DIR, track_line_gdf)

## §8. Exports + auto summary

In [ ]:
if PROCEED_GWR:
    for dv_col, dv_label in DVs:
        if gwr_results.get(dv_col):
            gm = gwr_results[dv_col]["model"]
            df_out = gdf_model[["cluster", "centroid_x", "centroid_y", "nchs_code"]].copy()
            df_out["local_R2"] = gm.localR2.flatten()
            for i, fn in enumerate(["intercept"] + FEATURES):
                df_out[f"beta_{fn}"] = gm.params[:, i]
                df_out[f"tval_{fn}"] = gm.tvalues[:, i]
            fn_out = os.path.join(OUTPUT_DIR, f"gwr_local_coefficients_{dv_col}.csv")
            df_out.to_csv(fn_out, index=False)
            print(f"Saved: {fn_out}")

# Quick textual summary
summary = [f"Helene GWR 100mi — N={len(gdf_model)}"]
summary.append(f"PROCEED_GWR (Moran's I or coord-residual p<0.10): {PROCEED_GWR}")
summary.append("\nGWR vs OLS AICc:")
summary.append(comp_df.to_string(index=False))
summary.append("\nKey comparison with 50mi run (see helene_gwr.ipynb):")
summary.append("  50mi: N=38, GWR bw=33–36/38 (near-global), MGWR FAILED")
summary.append(f"  100mi: N={len(gdf_model)}, GWR bw=see above")
with open(os.path.join(OUTPUT_DIR, "summary.txt"), "w") as f:
    f.write("\n".join(summary))
print("\n".join(summary))

## §9. Limitations to cite in the manuscript

1. **N=101 is adequate for GWR** but bandwidth uncertainty still nontrivial. Bootstrap of local coefficients (200 resamples) recommended before publishing local β maps.
2. **NCHS-homogeneous clustering** still partly absorbs the spatial heterogeneity GWR would detect. The 100mi cutoff adds Appalachian rural counties to the existing rural clusters, which doesn't necessarily increase the within-cluster signal.
3. **Cluster centroids not population-weighted.** Same caveat as the 50mi notebook.
4. **Euclidean centroid distance kernel** — same caveat. Hydrologic-exposure-similarity kernel is a future direction.
5. **Within-flow only.** Inflow/outflow GWR at 100mi can be added by adapting `helene_gwr_flow.ipynb`.

Once this notebook runs, update [findings.md](../notes/findings.md) §"Affected-region cutoff sensitivity" with the GWR coefficient localization result (or lack thereof). The most informative comparison: does the spatial heterogeneity of dist_to_track that I described in the 50mi run dissolve at 100mi (suggesting kernel-edge artefact), or does it persist (suggesting real two-regime structure)?